# 03 — The Bonsai HTTP surface

**Question this notebook answers:** what exactly crosses the boundary
between the controller and the PrismML llama.cpp server?

Bonsai is a reasoning model served by a llama.cpp fork. Next to the
OpenAI-compatible chat API it exposes `/tokenize` and `/detokenize` — the
bidirectional token access that makes the context a *proposal* rather than
a transcript. Companion reference: `docs/INTERACTION_MAP.md` §3.


In [1]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



project root: /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab


In [2]:
bonsai = BonsaiClient()
print("base url:", bonsai.base_url)
print("health:", bonsai.health(verbose=True))


base url: http://127.0.0.1:8081
GET http://127.0.0.1:8081/health
{
  "status": "ok"
}
health: True


## Tokenize / detokenize — the two directions

`tid{n}` memory becomes text via `/detokenize`; Bonsai's answer text
becomes `tid{n}` frames via `/tokenize`. These are plain llama.cpp
endpoints.


In [3]:
toks = bonsai.tokenize("The capital of France is Paris.", verbose=True)
print()
print("tokens:", toks)
print()
print("roundtrip:", repr(bonsai.detokenize(toks, verbose=True)))


POST http://127.0.0.1:8081/tokenize
{
  "content": "The capital of France is Paris."
}
{
  "tokens": [
    760,
    6511,
    314,
    9338,
    369,
    11751,
    13
  ]
}

tokens: [760, 6511, 314, 9338, 369, 11751, 13]

POST http://127.0.0.1:8081/detokenize
{
  "tokens": [
    760,
    6511,
    314,
    9338,
    369,
    11751,
    13
  ]
}
{
  "content": "The capital of France is Paris."
}
roundtrip: 'The capital of France is Paris.'


## Chat — the OpenAI-compatible endpoint with reasoning

Bonsai emits `reasoning_content` (its thinking trace) before `content`
(the answer). `usage.prompt_tokens` is the honest per-turn context cost
the controller records.


In [4]:
full = bonsai.chat_full(
    "What is the capital of France? Answer in one short sentence.",
    max_tokens=64, temperature=0.0, verbose=True,
)
print()
print("answer:", repr(full["content"]))
print("reasoning:", repr(full["reasoning"][:200]))
print("usage:", full["usage"])


POST http://127.0.0.1:8081/v1/chat/completions
{
  "messages": [
    {
      "role": "user",
      "content": "What is the capital of France? Answer in one short sentence."
    }
  ],
  "max_tokens": 64,
  "temperature": 0.0,
  "stream": false
}
{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The capital of France is Paris.",
        "reasoning_content": "We need to answer user's request: \"What is the capital of France? Answer in one short sentence.\" Simple. Final one short sentence.\n"
      }
    }
  ],
  "created": 1790185593,
  "model": "models/bonsai2-gguf/27B/Ternary-Bonsai-2-27B-PQ2_0.gguf",
  "system_fingerprint": "b10709-9a9394a89",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 40,
    "prompt_tokens": 65,
    "total_tokens": 105,
    "prompt_tokens_details": {
      "cached_tokens": 61
    }
  },
  "id": "chatcmpl-yf8AXBbzNevsEKdN4h8sC2KlFEVdG25Y",
  "timings":

## If the server is DOWN

If the cell above raised `URLError`, start Bonsai in another terminal:

```bash
cd ~/tools/Bonsai-demo
LD_LIBRARY_PATH="$PWD/bin/cuda" ./bin/cuda/llama-server \
  -m models/bonsai2-gguf/27B/Ternary-Bonsai-2-27B-PQ2_0.gguf \
  -ngl 99 -fa on -c 2048 --host 127.0.0.1 --port 8081
```

Do **not** set `CUDA_VISIBLE_DEVICES` — the PrismML fork lists the RTX
3060 as CUDA0, opposite to `nvidia-smi`.
